# Resource Idle Time Analysis

Metric definition (per timestamp, per activity):
- If `queue >= resource` => `idle = 0`
- If `queue < resource` => `idle = resource - queue`

Then sum over all activities and all timestamps to get `total_idle_time`.
Finally normalize by total resources (sum of staff across activities): `idle_per_resource = total_idle_time / total_resources`.

## Workflow:
1. **Random baseline**: Load existing queue log directly
2. **6 iterative models** (DQN, DDQN, Dueling, PerDQN, MultiStepDQN, Rainbow): Specify (algo, gen) → load event log → reconstruct queue log → compute idle time
3. **2 offline models** (FORLAPS, LearningToAct): Load event log → reconstruct queue log → compute idle time

In [1]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import pandas as pd
import numpy as np
from common.queue_log_utils import reconstruct_queue_log_from_event_log

# Load activity info (for staff counts)
activity_info_path = Path('../data/raw/activity_info.json')
activity_info = json.loads(activity_info_path.read_text(encoding='utf-8'))
activity_columns = sorted(activity_info.keys(), key=lambda x: activity_info[x]['id'])
staff_mapping = {name: int(info['staff']) for name, info in activity_info.items()}
staff_array = np.array([staff_mapping[act] for act in activity_columns], dtype=float)
total_resources = float(staff_array.sum())

print(f'Total resources (sum staff): {total_resources}')
print(f'Activities: {len(activity_columns)}')

Total resources (sum staff): 50.0
Activities: 21


In [2]:
def compute_idle_time_from_queue_log(df_queue, staff_array, activity_columns):
    """
    Compute idle time metrics from queue log DataFrame.
    
    Returns: dict with total_idle_time, idle_per_resource, and breakdown DataFrames
    """
    queue_matrix = df_queue[activity_columns].to_numpy(dtype=float)
    idle_matrix = np.maximum(staff_array - queue_matrix, 0.0)
    idle_per_timestamp = idle_matrix.sum(axis=1)
    
    total_idle_time = float(idle_per_timestamp.sum())
    total_res = float(staff_array.sum())
    idle_per_resource = total_idle_time / total_res if total_res else float('nan')
    
    # Per-activity breakdown
    idle_per_activity = idle_matrix.sum(axis=0)
    df_idle_act = pd.DataFrame({
        'Activity': activity_columns,
        'Staff': staff_array,
        'TotalIdle': idle_per_activity,
        'IdlePerResource': np.where(staff_array > 0, idle_per_activity / staff_array, np.nan),
    }).sort_values('TotalIdle', ascending=False)
    
    return {
        'total_idle_time': total_idle_time,
        'total_resources': total_res,
        'idle_per_resource': idle_per_resource,
        'avg_idle_per_timestamp': float(idle_per_timestamp.mean()),
        'breakdown': df_idle_act
    }

print('Helper function defined: compute_idle_time_from_queue_log')

Helper function defined: compute_idle_time_from_queue_log


In [3]:
# ============================================================
# 1. RANDOM BASELINE (load existing queue log directly)
# ============================================================
queue_log_path = Path('../data/raw/200_queue_log_version_random_base.csv')
df_queue_random = pd.read_csv(queue_log_path)

result_random = compute_idle_time_from_queue_log(df_queue_random, staff_array, activity_columns)

print('=== RANDOM BASELINE ===')
print(f"Total Idle Time: {result_random['total_idle_time']:.2f}")
print(f"Idle per Resource: {result_random['idle_per_resource']:.2f}")
print(f"Avg Idle per Timestamp: {result_random['avg_idle_per_timestamp']:.2f}")
print()
result_random['breakdown'].head(5)

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\200_queue_log_version_random_base.csv'

In [ ]:
# ============================================================
# 2. ITERATIVE MODELS (6 models with generation selection)
# ============================================================
# Specify model and generation to analyze
# Example: ("DQN", 2) will load event log for DQN Gen 2

ITERATIVE_MODELS = ["DQN", "DDQN", "Dueling", "PerDQN", "MultiStepDQN", "Rainbow"]

def compute_idle_for_iterative_model(algo_name, gen_id):
    """
    Load event log for a specific (algo, generation), reconstruct queue log, compute idle time.
    
    Event log naming convention: data/evaluate/event_log_<algo_tag>_version_<gen_id>.csv
    where <algo_tag> is lowercase algo name (e.g., 'dqn', 'ddqn')
    """
    algo_tag = algo_name.lower()
    event_log_path = Path(f'../data/evaluate/event_log_{algo_tag}_version_{gen_id}.csv')
    
    if not event_log_path.exists():
        # Try alternative naming: some models may use different tags
        # Fallback: try model_version pattern
        event_log_path = Path(f'../data/evaluate/event_log_model_version_{gen_id}.csv')
        if not event_log_path.exists():
            return None, f"Event log not found for {algo_name} Gen {gen_id}"
    
    # Reconstruct queue log from event log
    df_queue = reconstruct_queue_log_from_event_log(str(event_log_path), '../data/raw/activity_info.json')
    
    # Compute idle time
    result = compute_idle_time_from_queue_log(df_queue, staff_array, activity_columns)
    result['algo'] = algo_name
    result['gen'] = gen_id
    
    return result, None

# Example: Compute for DQN Gen 2
# Uncomment and modify to analyze specific model/generation:
# result, error = compute_idle_for_iterative_model("DQN", 2)
# if error:
#     print(f"Error: {error}")
# else:
#     print(f"=== {result['algo']} Gen {result['gen']} ===")
#     print(f"Idle per Resource: {result['idle_per_resource']:.2f}")
#     display(result['breakdown'].head(5))

print("Function defined: compute_idle_for_iterative_model(algo_name, gen_id)")

In [ ]:
# ============================================================
# 3. OFFLINE MODELS (FORLAPS and LearningToAct)
# ============================================================
# These models don't have generations, just a single trained policy

def compute_idle_for_offline_model(algo_name):
    """
    Load event log for offline model, reconstruct queue log, compute idle time.
    
    Event log naming convention: data/evaluate/event_log_<algo_tag>_version_<version>.csv
    For FORLAPS: event_log_forlaps_version_forlaps.csv or similar
    For LearningToAct: event_log_learningtoact_version_learningtoact.csv or similar
    """
    algo_tag = algo_name.lower()
    
    # Try common naming patterns
    possible_paths = [
        Path(f'../data/evaluate/event_log_{algo_tag}_version_{algo_tag}.csv'),
        Path(f'../data/evaluate/event_log_version_{algo_tag}.csv'),
        Path(f'../data/evaluate/event_log_{algo_tag}.csv'),
    ]
    
    event_log_path = None
    for p in possible_paths:
        if p.exists():
            event_log_path = p
            break
    
    if event_log_path is None:
        return None, f"Event log not found for {algo_name}"
    
    # Reconstruct queue log from event log
    df_queue = reconstruct_queue_log_from_event_log(str(event_log_path), '../data/raw/activity_info.json')
    
    # Compute idle time
    result = compute_idle_time_from_queue_log(df_queue, staff_array, activity_columns)
    result['algo'] = algo_name
    
    return result, None

# Example: Compute for FORLAPS
# Uncomment to analyze:
# result, error = compute_idle_for_offline_model("FORLAPS")
# if error:
#     print(f"Error: {error}")
# else:
#     print(f"=== {result['algo']} ===")
#     print(f"Idle per Resource: {result['idle_per_resource']:.2f}")
#     display(result['breakdown'].head(5))

print("Function defined: compute_idle_for_offline_model(algo_name)")

In [ ]:
# ============================================================
# 4. INTERACTIVE ANALYSIS - Specify your model here
# ============================================================
# Change these variables to analyze different models:

# For iterative models (6 models):
ALGO_NAME = "DQN"  # Options: "DQN", "DDQN", "Dueling", "PerDQN", "MultiStepDQN", "Rainbow"
GEN_ID = 2         # Generation number (1-5)

result, error = compute_idle_for_iterative_model(ALGO_NAME, GEN_ID)
if error:
    print(f"❌ Error: {error}")
else:
    print(f"=== {result['algo']} Gen {result['gen']} ===")
    print(f"Total Idle Time: {result['total_idle_time']:.2f}")
    print(f"Idle per Resource: {result['idle_per_resource']:.2f}")
    print(f"Avg Idle per Timestamp: {result['avg_idle_per_timestamp']:.2f}")
    print("\nTop 5 activities by total idle time:")
    display(result['breakdown'].head(5))

In [ ]:
# For offline models (FORLAPS or LearningToAct):
# Uncomment to analyze offline models:

# OFFLINE_ALGO = "FORLAPS"  # Options: "FORLAPS", "LearningToAct"
# result, error = compute_idle_for_offline_model(OFFLINE_ALGO)
# if error:
#     print(f"❌ Error: {error}")
# else:
#     print(f"=== {result['algo']} ===")
#     print(f"Total Idle Time: {result['total_idle_time']:.2f}")
#     print(f"Idle per Resource: {result['idle_per_resource']:.2f}")
#     print(f"Avg Idle per Timestamp: {result['avg_idle_per_timestamp']:.2f}")
#     print("\nTop 5 activities by total idle time:")
#     display(result['breakdown'].head(5))

In [ ]:
# ============================================================
# 5. BATCH ANALYSIS - Compare all models
# ============================================================
# Compute idle time for all models and create comparison table

results_all = []

# Random baseline
result_random['algo'] = 'Random'
result_random['gen'] = 'baseline'
results_all.append({
    'Model': 'Random',
    'Generation': 'baseline',
    'Total_Idle_Time': result_random['total_idle_time'],
    'Idle_per_Resource': result_random['idle_per_resource'],
    'Avg_Idle_per_Timestamp': result_random['avg_idle_per_timestamp']
})

# 6 iterative models - try all generations
for algo in ITERATIVE_MODELS:
    for gen in range(1, 6):  # Gen 1-5
        result, error = compute_idle_for_iterative_model(algo, gen)
        if result and not error:
            results_all.append({
                'Model': algo,
                'Generation': f'Gen{gen}',
                'Total_Idle_Time': result['total_idle_time'],
                'Idle_per_Resource': result['idle_per_resource'],
                'Avg_Idle_per_Timestamp': result['avg_idle_per_timestamp']
            })

# 2 offline models
for offline_algo in ["FORLAPS", "LearningToAct"]:
    result, error = compute_idle_for_offline_model(offline_algo)
    if result and not error:
        results_all.append({
            'Model': offline_algo,
            'Generation': 'offline',
            'Total_Idle_Time': result['total_idle_time'],
            'Idle_per_Resource': result['idle_per_resource'],
            'Avg_Idle_per_Timestamp': result['avg_idle_per_timestamp']
        })

# Create comparison DataFrame
df_comparison = pd.DataFrame(results_all)
print(f"Total models analyzed: {len(df_comparison)}")
df_comparison.head(20)